This language model calculates probability of the next token based on the past and current tokens(causal system).
In this case we use the complete plays of Shakespear and work ar the character level, so the vocabulary is 65 symboles and training takes minutes 

In [6]:
import os
import urllib.request
import torch
from torch import nn
import matplotlib.pyplot as plt
%matplotlib inline

url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
if not os.path.exists("tinyshakespeare.txt"):
    urllib.request.urlretrieve(url, "tinyshakespeare.txt")
    text = open("tinyshakespeare.txt", encoding="utf-8").read()

chars = sorted(set(text))
vocab_size = len(chars)

 #encoding chars to numbers
stoi= {c: i for i,c in enumerate(chars)}

#decoding numbers to chars
itos= {i: c for c,i in stoi.items()}

data = torch.tensor([stoi[c] for c in text], dtype = torch.long)
#90% of data is used for training and the rest is for the validation 
split = int(0.9 * len(data))
train_data, val_data = data[:split], data[split:]

print("chars:", len(text), "| vocabulary:", vocab_size)
print("sample:", repr(text[:90]))

chars: 1115394 | vocabulary: 65
sample: 'First Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Ci'


A token embedding table turns each chars id into a vector.
A position embedding adds the position, because attention only is order-blind.
then 4 transformer layers, also a causal mask is now switched on so no position can read the future(by setting the future vals equal to -infinity, when the vector is inserted as an input to the softMax the output would be zero)

In [ ]:
BLOCK, BATCH, D_MODEL, HEADS, LAYERS = 128, 64, 128, 4, 4

class TransformerLayer(nn.Module):
    def __init__(self, d_model, heads):
        super().__init__()
        self.attention = nn.MultiheadAttention(d_model, heads, batch_first=True)
        self.feed_forward = nn.Sequential(
            nn.Linear(d_model, 4 * d_model), nn.ReLU(), nn.Linear(4 * d_model, d_model))
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x, mask):
        h = self.norm1(x)
        x = x + self.attention(h, h, h, attn_mask=mask, need_weights=False)[0]
        return x + self.feed_forward(self.norm2(x))
